# Data Overview

This notebook inspects the parquet datasets under `experiments/data/parquets` before they are used in experiments.

It focuses on:
- dataset discovery
- file-level row and size summaries
- schema comparison between green and yellow taxi data
- a few quick quality checks for experiment readiness


In [1]:
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds
import pyarrow.parquet as pq

DATA_ROOT = Path("experiments/data/parquets")
DATASETS = sorted([path for path in DATA_ROOT.iterdir() if path.is_dir()])

print(f"Data root: {DATA_ROOT.resolve()}")
print("Dataset folders:", [path.name for path in DATASETS])

Data root: /Users/andrew/Dev/Projects/DataGate/experiments/data/parquets
Dataset folders: ['green', 'yellow']


In [2]:
overview_rows = []
for dataset_dir in DATASETS:
    files = sorted(dataset_dir.glob("*.parquet"))
    total_rows = 0
    total_size_mb = 0.0
    for file_path in files:
        parquet_file = pq.ParquetFile(file_path)
        total_rows += parquet_file.metadata.num_rows
        total_size_mb += file_path.stat().st_size / 1024 / 1024
    overview_rows.append(
        {
            "dataset": dataset_dir.name,
            "files": len(files),
            "total_rows": total_rows,
            "total_size_mb": round(total_size_mb, 2),
            "first_file": files[0].name if files else None,
            "last_file": files[-1].name if files else None,
        }
    )

overview_df = pd.DataFrame(overview_rows)
overview_df

   dataset  files     total_rows  total_size_mb                              first_file                               last_file
0    green     14        669020          15.61   green_tripdata_2025-01.parquet   green_tripdata_2026-02.parquet
1   yellow     14      55847357         908.68  yellow_tripdata_2025-01.parquet  yellow_tripdata_2026-02.parquet

In [3]:
file_rows = []
for dataset_dir in DATASETS:
    for file_path in sorted(dataset_dir.glob("*.parquet")):
        parquet_file = pq.ParquetFile(file_path)
        file_rows.append(
            {
                "dataset": dataset_dir.name,
                "file": file_path.name,
                "rows": parquet_file.metadata.num_rows,
                "size_mb": round(file_path.stat().st_size / 1024 / 1024, 2),
            }
        )

file_summary_df = pd.DataFrame(file_rows)
file_summary_df

   dataset                           file     rows  size_mb
0    green   green_tripdata_2025-01.parquet    48326     1.12
1    green   green_tripdata_2025-02.parquet    46621     1.07
2    green   green_tripdata_2025-03.parquet    51539     1.20
3    green   green_tripdata_2025-04.parquet    52132     1.21
4    green   green_tripdata_2025-05.parquet    55399     1.28
5    green   green_tripdata_2025-06.parquet    49390     1.16
6    green   green_tripdata_2025-07.parquet    48205     1.13
7    green   green_tripdata_2025-08.parquet    46306     1.10
8    green   green_tripdata_2025-09.parquet    48893     1.15
9    green   green_tripdata_2025-10.parquet    49416     1.15
10   green   green_tripdata_2025-11.parquet    46912     1.11
11   green   green_tripdata_2025-12.parquet    48236     1.12
12   green   green_tripdata_2026-01.parquet    40272     0.95
13   green   green_tripdata_2026-02.parquet    37373     0.88
14  yellow  yellow_tripdata_2025-01.parquet  3475226    56.42
15  yellow

In [4]:
schemas = {}
for dataset_dir in DATASETS:
    first_file = sorted(dataset_dir.glob("*.parquet"))[0]
    schema = pq.ParquetFile(first_file).schema_arrow
    schemas[dataset_dir.name] = {field.name: str(field.type) for field in schema}

all_columns = sorted(set().union(*[schema.keys() for schema in schemas.values()]))
schema_comparison_df = pd.DataFrame(
    {
        "column": all_columns,
        "green_type": [schemas.get("green", {}).get(column) for column in all_columns],
        "yellow_type": [schemas.get("yellow", {}).get(column) for column in all_columns],
    }
)
schema_comparison_df

                column            green_type           yellow_type
0             Airport_fee                  None               double
1            DOLocationID                 int32                int32
2             PULocationID                 int32                int32
3              RatecodeID                 int64                int64
4                VendorID                 int32                int32
5      cbd_congestion_fee                double               double
6   congestion_surcharge                double               double
7              ehail_fee                double                 None
8                  extra                double               double
9            fare_amount                double               double
10  improvement_surcharge                double               double
11    lpep_dropoff_datetime         timestamp[us]                 None
12     lpep_pickup_datetime         timestamp[us]                 None
13                mta_tax          

In [5]:
quality_configs = {
    "green": {"datetime": "lpep_pickup_datetime"},
    "yellow": {"datetime": "tpep_pickup_datetime"},
}

quality_rows = []
for dataset_name, config in quality_configs.items():
    dataset = ds.dataset(str(DATA_ROOT / dataset_name), format="parquet")
    table = dataset.to_table(columns=[config["datetime"], "trip_distance", "total_amount"])
    frame = table.to_pandas()
    quality_rows.append(
        {
            "dataset": dataset_name,
            "pickup_min": frame[config["datetime"]].min(),
            "pickup_max": frame[config["datetime"]].max(),
            "trip_distance_mean": round(float(frame["trip_distance"].mean()), 3),
            "trip_distance_p95": round(float(frame["trip_distance"].quantile(0.95)), 3),
            "total_amount_mean": round(float(frame["total_amount"].mean()), 3),
            "total_amount_min": round(float(frame["total_amount"].min()), 2),
            "total_amount_max": round(float(frame["total_amount"].max()), 2),
        }
    )

quality_df = pd.DataFrame(quality_rows)
quality_df

   dataset          pickup_min          pickup_max  trip_distance_mean  trip_distance_p95  total_amount_mean  total_amount_min  total_amount_max
0    green 2008-12-31 15:13:04 2026-03-01 09:48:53              18.772               9.81             25.092           -878.00         620592.77
1   yellow 2007-12-05 18:45:00 2026-03-01 00:51:48               6.780              12.69             27.255           -426.00          83485.66

## Notes Before Experiments

- `yellow` is much larger than `green` and will dominate runtime and memory if both are loaded eagerly.
- Pickup timestamp ranges include records from 2007-2008 even though the files are labeled 2025-2026. That is a strong signal to add date filtering or outlier handling in experiment pipelines.
- `green` and `yellow` do not share identical schemas. `green` has `trip_type` and `ehail_fee`; `yellow` has `Airport_fee`; pickup and dropoff datetime column names also differ.
- `total_amount` contains negative values and very large positive outliers, so downstream modeling should validate fare-related columns before training.
